In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

In [ ]:
import os
import time
import random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

In [ ]:
print("train_pairs" in globals())
print("val_pairs" in globals())

In [ ]:
%pip install opencv-python

In [ ]:
%pip install opencv-python-headless

In [ ]:
# Create the spatial attention module
attention = SpatialAttention()

# Create a sample input tensor
x = torch.randn(1, 16, 256, 256)

# Apply spatial attention
output = attention(x)

# Print input and output shapes
print("Input shape:", x.shape)
print("Output shape:", output.shape)

In [ ]:
####Input shape: torch.Size([1, 16, 256, 256])
Output shape: torch.Size([1, 16, 256, 256])

In [8]:
import torch
import torch.nn as nn


In [16]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

import os
import time
import random
import numpy as np
import torch

DATASET_ROOT = r"C:\Users\HP\Downloads\archive (10)\brisc2025"

TRAIN_IMAGE_DIR = os.path.join(DATASET_ROOT, "segmentation_task", "train", "images")
TRAIN_MASK_DIR = os.path.join(DATASET_ROOT, "segmentation_task", "train", "masks")
TEST_IMAGE_DIR = os.path.join(DATASET_ROOT, "segmentation_task", "test", "images")
TEST_MASK_DIR = os.path.join(DATASET_ROOT, "segmentation_task", "test", "masks")

IMAGE_SIZE = 128
TRAIN_SAMPLES = 1000
VAL_SAMPLES = 200

BATCH_SIZE = 8
EPOCHS = 15
LEARNING_RATE = 0.001

NUM_WORKERS = 0
SEED = 42

MODEL_PATH = os.path.join(DATASET_ROOT, "lightweight_attention_best.pth")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_num_threads(4)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Train/Val samples:", TRAIN_SAMPLES, "/", VAL_SAMPLES)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)

Device: cpu
Image size: 128
Train/Val samples: 1000 / 200
Batch size: 8
Epochs: 15


In [19]:
import time
import torch
import torch.nn as nn
import torch.optim as optim

# ---- Check that earlier cells have been run ----
for name in ["DEVICE", "LEARNING_RATE", "EPOCHS", "MODEL_PATH", "train_loader", "val_loader"]:
    if name not in globals():
        raise NameError(
            f"'{name}' is missing. Run the config cell and the DataLoaders cell first."
        )


# ---- Model ----
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class AttentionGate(nn.Module):
    def __init__(self, g_c, x_c, inter_c):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(g_c, inter_c, 1), nn.BatchNorm2d(inter_c))
        self.Wx = nn.Sequential(nn.Conv2d(x_c, inter_c, 1), nn.BatchNorm2d(inter_c))
        self.psi = nn.Sequential(nn.Conv2d(inter_c, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        a = self.relu(self.Wg(g) + self.Wx(x))
        return x * self.psi(a)


class LightweightAttentionUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)
        self.bottleneck = ConvBlock(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.att4 = AttentionGate(base * 8, base * 8, base * 4)
        self.dec4 = ConvBlock(base * 16, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.att3 = AttentionGate(base * 4, base * 4, base * 2)
        self.dec3 = ConvBlock(base * 8, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.att2 = AttentionGate(base * 2, base * 2, base)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.att1 = AttentionGate(base, base, base // 2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([self.att4(d4, e4), d4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([self.att3(d3, e3), d3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([self.att2(d2, e2), d2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([self.att1(d1, e1), d1], dim=1))

        return self.out(d1)  # raw logits


model = LightweightAttentionUNet().to(DEVICE)
print("Model created. Parameters:", sum(p.numel() for p in model.parameters()))


# ---- Training ----
def dice_per_batch(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(1, 2, 3))
    total = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return (2 * inter + eps) / (total + eps)


bce = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

best_val_dice = 0.0

for epoch in range(EPOCHS):
    start = time.time()

    # Train
    model.train()
    train_loss = 0.0
    for images, masks, _ in train_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = bce(logits, masks) + (1 - dice_per_batch(logits, masks).mean())
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validate
    model.eval()
    val_dice = 0.0
    with torch.no_grad():
        for images, masks, _ in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            val_dice += dice_per_batch(logits, masks).mean().item()

    train_loss /= len(train_loader)
    val_dice /= len(val_loader)
    scheduler.step(val_dice)

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), MODEL_PATH)
        saved = " (saved)"
    else:
        saved = ""

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"{time.time() - start:.0f}s{saved}"
    )

print("Best val Dice:", round(best_val_dice, 4))

NameError: 'train_loader' is missing. Run the config cell and the DataLoaders cell first.

In [21]:
# ============================================================
# DATA: PAIRS + SPLIT + DATASET + DATALOADERS (all in one)
# ============================================================

import os
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# ---- Check that the config cell has been run ----
needed = [
    "TRAIN_IMAGE_DIR", "TRAIN_MASK_DIR",
    "TEST_IMAGE_DIR", "TEST_MASK_DIR",
    "IMAGE_SIZE", "TRAIN_SAMPLES", "VAL_SAMPLES",
    "BATCH_SIZE", "NUM_WORKERS", "SEED",
]
for name in needed:
    if name not in globals():
        raise NameError(name + " is missing. Run the config cell first.")

EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")


# ---- 1. Match images with masks ----
def create_pairs(image_dir, mask_dir):
    image_files = sorted(
        f for f in os.listdir(image_dir)
        if f.lower().endswith(EXTENSIONS)
    )
    mask_files = sorted(
        f for f in os.listdir(mask_dir)
        if f.lower().endswith(EXTENSIONS)
    )
    mask_dict = {os.path.splitext(f)[0]: f for f in mask_files}

    pairs = []
    for image_file in image_files:
        stem = os.path.splitext(image_file)[0]
        if stem not in mask_dict:
            continue

        name_lower = stem.lower()
        if "_gl_" in name_lower:
            tumor_type = "Glioma"
        elif "_me_" in name_lower:
            tumor_type = "Meningioma"
        elif "_pi_" in name_lower:
            tumor_type = "Pituitary"
        elif "_nt_" in name_lower:
            tumor_type = "No Tumor"
        else:
            continue

        image_path = os.path.join(image_dir, image_file)
        mask_path = os.path.join(mask_dir, mask_dict[stem])
        pairs.append((image_path, mask_path, tumor_type))

    return pairs


all_train_pairs = create_pairs(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR)
test_pairs = create_pairs(TEST_IMAGE_DIR, TEST_MASK_DIR)

print("Available training pairs:", len(all_train_pairs))
print("Available test pairs:", len(test_pairs))

total_needed = TRAIN_SAMPLES + VAL_SAMPLES
if len(all_train_pairs) < total_needed:
    raise ValueError("Not enough training images.")


# ---- 2. Train / validation split (training folder only) ----
labels = [p[2] for p in all_train_pairs]

train_pairs, val_pairs = train_test_split(
    all_train_pairs,
    train_size=TRAIN_SAMPLES,
    test_size=VAL_SAMPLES,
    random_state=SEED,
    stratify=labels,
)

print("Training samples:", len(train_pairs))
print("Validation samples:", len(val_pairs))


# ---- 3. Dataset class ----
class BrainTumorDataset(Dataset):

    def __init__(self, pairs, image_size=128):
        self.pairs = pairs
        self.image_size = image_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path, tumor_type = self.pairs[index]
        size = (self.image_size, self.image_size)

        image = Image.open(image_path).convert("L")
        image = image.resize(size, Image.Resampling.BILINEAR)
        image = np.asarray(image, dtype=np.float32) / 255.0

        mask = Image.open(mask_path).convert("L")
        mask = mask.resize(size, Image.Resampling.NEAREST)
        mask = np.asarray(mask, dtype=np.uint8)
        mask = (mask > 127).astype(np.float32)

        image = torch.from_numpy(image.copy()).unsqueeze(0)
        mask = torch.from_numpy(mask.copy()).unsqueeze(0)

        return image, mask, tumor_type


# ---- 4. DataLoaders ----
train_dataset = BrainTumorDataset(train_pairs, IMAGE_SIZE)
val_dataset = BrainTumorDataset(val_pairs, IMAGE_SIZE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
)

print("\nDataLoaders created successfully.")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

images, masks, types = next(iter(train_loader))
print("Image batch:", images.shape)
print("Mask batch:", masks.shape)
print("Tumor types:", types)

Available training pairs: 3933
Available test pairs: 860
Training samples: 1000
Validation samples: 200

DataLoaders created successfully.
Train batches: 125
Val batches: 200
Image batch: torch.Size([8, 1, 128, 128])
Mask batch: torch.Size([8, 1, 128, 128])
Tumor types: ('Pituitary', 'Pituitary', 'Meningioma', 'Pituitary', 'Glioma', 'Meningioma', 'Meningioma', 'Meningioma')


In [23]:
import os
import time
import random
import numpy as np
import torch

DATASET_ROOT = r"C:\Users\HP\Downloads\archive (10)\brisc2025"

seg = os.path.join(DATASET_ROOT, "segmentation_task")
TRAIN_IMAGE_DIR = os.path.join(seg, "train", "images")
TRAIN_MASK_DIR = os.path.join(seg, "train", "masks")
TEST_IMAGE_DIR = os.path.join(seg, "test", "images")
TEST_MASK_DIR = os.path.join(seg, "test", "masks")

# ---- Light settings (raise later if your laptop stays cool) ----
IMAGE_SIZE = 128
TRAIN_SAMPLES = 400
VAL_SAMPLES = 80
BATCH_SIZE = 4
EPOCHS = 10
LEARNING_RATE = 0.001
COOLDOWN_SECONDS = 15   # pause between epochs so the CPU cools down
RESUME = True           # continue from the last finished epoch

NUM_WORKERS = 0
SEED = 42

MODEL_PATH = os.path.join(DATASET_ROOT, "sabernet_best.pth")
CKPT_PATH = os.path.join(DATASET_ROOT, "sabernet_checkpoint.pth")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_num_threads(2)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)
print("Train/Val:", TRAIN_SAMPLES, "/", VAL_SAMPLES)
print("Batch:", BATCH_SIZE, "| Epochs:", EPOCHS)

Device: cpu
Train/Val: 400 / 80
Batch: 4 | Epochs: 10


In [24]:
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")


def create_pairs(image_dir, mask_dir):
    image_files = sorted(
        f for f in os.listdir(image_dir) if f.lower().endswith(EXTENSIONS)
    )
    mask_files = sorted(
        f for f in os.listdir(mask_dir) if f.lower().endswith(EXTENSIONS)
    )
    mask_dict = {os.path.splitext(f)[0]: f for f in mask_files}

    pairs = []
    for image_file in image_files:
        stem = os.path.splitext(image_file)[0]
        if stem not in mask_dict:
            continue

        name = stem.lower()
        if "_gl_" in name:
            tumor_type = "Glioma"
        elif "_me_" in name:
            tumor_type = "Meningioma"
        elif "_pi_" in name:
            tumor_type = "Pituitary"
        elif "_nt_" in name:
            tumor_type = "No Tumor"
        else:
            continue

        image_path = os.path.join(image_dir, image_file)
        mask_path = os.path.join(mask_dir, mask_dict[stem])
        pairs.append((image_path, mask_path, tumor_type))
    return pairs


class BrainTumorDataset(Dataset):
    """Loads every image ONCE into memory (small uint8 arrays)."""

    def __init__(self, pairs, image_size=128):
        size = (image_size, image_size)
        self.types = [p[2] for p in pairs]
        self.images = []
        self.masks = []
        for image_path, mask_path, _ in pairs:
            img = Image.open(image_path).convert("L")
            img = img.resize(size, Image.Resampling.BILINEAR)
            msk = Image.open(mask_path).convert("L")
            msk = msk.resize(size, Image.Resampling.NEAREST)
            self.images.append(np.asarray(img, dtype=np.uint8))
            self.masks.append(np.asarray(msk, dtype=np.uint8) > 127)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        image = self.images[i].astype(np.float32) / 255.0
        mask = self.masks[i].astype(np.float32)
        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)
        return image, mask, self.types[i]


all_train_pairs = create_pairs(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR)
test_pairs = create_pairs(TEST_IMAGE_DIR, TEST_MASK_DIR)
print("Train pool:", len(all_train_pairs), "| Test pairs:", len(test_pairs))

labels = [p[2] for p in all_train_pairs]
train_pairs, val_pairs = train_test_split(
    all_train_pairs,
    train_size=TRAIN_SAMPLES,
    test_size=VAL_SAMPLES,
    random_state=SEED,
    stratify=labels,
)

train_dataset = BrainTumorDataset(train_pairs, IMAGE_SIZE)
val_dataset = BrainTumorDataset(val_pairs, IMAGE_SIZE)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS,
)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))

Train pool: 3933 | Test pairs: 860
Train batches: 100 | Val batches: 20


In [28]:
import torch.nn as nn
import torch.optim as optim


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class AttentionGate(nn.Module):
    def __init__(self, g_c, x_c, inter_c):
        super().__init__()
        self.Wg = nn.Sequential(
            nn.Conv2d(g_c, inter_c, 1), nn.BatchNorm2d(inter_c)
        )
        self.Wx = nn.Sequential(
            nn.Conv2d(x_c, inter_c, 1), nn.BatchNorm2d(inter_c)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_c, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        a = self.relu(self.Wg(g) + self.Wx(x))
        return x * self.psi(a)


class LightweightAttentionUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=8):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)
        self.bottleneck = ConvBlock(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.att4 = AttentionGate(base * 8, base * 8, base * 4)
        self.dec4 = ConvBlock(base * 16, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.att3 = AttentionGate(base * 4, base * 4, base * 2)
        self.dec3 = ConvBlock(base * 8, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.att2 = AttentionGate(base * 2, base * 2, base)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.att1 = AttentionGate(base, base, max(base // 2, 1))
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([self.att4(d4, e4), d4], dim=1))
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([self.att3(d3, e3), d3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([self.att2(d2, e2), d2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([self.att1(d1, e1), d1], dim=1))
        return self.out(d1)


def dice_per_batch(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(1, 2, 3))
    total = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return (2 * inter + eps) / (total + eps)


model = LightweightAttentionUNet(base=8).to(DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))

bce = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

start_epoch = 0
best_val_dice = 0.0

if RESUME and os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
    best_val_dice = ckpt["best_val_dice"]
    print("Resuming from epoch", start_epoch + 1)

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()

    model.train()
    train_loss = 0.0
    for images, masks, _ in train_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        dice_loss = 1 - dice_per_batch(logits, masks).mean()
        loss = bce(logits, masks) + dice_loss
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_dice = 0.0
    with torch.no_grad():
        for images, masks, _ in val_loader:
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)
            val_dice += dice_per_batch(model(images), masks).mean().item()

    train_loss /= len(train_loader)
    val_dice /= len(val_loader)

    note = ""
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), MODEL_PATH)
        note = " (best saved)"

    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch,
        "best_val_dice": best_val_dice,
    }, CKPT_PATH)

    print(
        "Epoch", epoch + 1, "/", EPOCHS,
        "| loss", round(train_loss, 4),
        "| val dice", round(val_dice, 4),
        "|", round(time.time() - t0), "s", note,
    )
    time.sleep(COOLDOWN_SECONDS)

print("Training finished. Best val Dice:", round(best_val_dice, 4))

Parameters: 492281
Resuming from epoch 11
Training finished. Best val Dice: 0.4774


In [29]:
import pandas as pd
from scipy.ndimage import distance_transform_edt, binary_erosion
from IPython.display import display

THRESHOLD = 0.5
CLASSES = ["Glioma", "Meningioma", "Pituitary"]
MAX_TEST_PER_CLASS = None   # e.g. 100 to evaluate faster; None = use all

# ---- Load best weights ----
model = LightweightAttentionUNet(base=8).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# ---- Keep only tumor images (No Tumor has empty masks) ----
selected = []
for c in CLASSES:
    items = [p for p in test_pairs if p[2] == c]
    if MAX_TEST_PER_CLASS:
        items = items[:MAX_TEST_PER_CLASS]
    selected += items

test_dataset = BrainTumorDataset(selected, IMAGE_SIZE)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
print("Test images used:", len(test_dataset))


def iou_score(pred, gt):
    union = np.logical_or(pred, gt).sum()
    return np.logical_and(pred, gt).sum() / union if union else 1.0


def dice_score(pred, gt):
    total = pred.sum() + gt.sum()
    inter = np.logical_and(pred, gt).sum()
    return 2 * inter / total if total else 1.0


def surface(mask):
    return mask & ~binary_erosion(mask)


def hd95_score(pred, gt):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    ps, gs = surface(pred), surface(gt)
    d_to_gt = distance_transform_edt(~gs)
    d_to_pred = distance_transform_edt(~ps)
    both = np.concatenate([d_to_gt[ps], d_to_pred[gs]])
    return np.percentile(both, 95)


records = []
with torch.no_grad():
    for images, masks, types in test_loader:
        probs = torch.sigmoid(model(images.to(DEVICE)))
        preds = (probs > THRESHOLD).cpu().numpy()[:, 0].astype(bool)
        gts = masks.numpy()[:, 0].astype(bool)
        for pred, gt, t in zip(preds, gts, types):
            records.append({
                "type": t,
                "iou": iou_score(pred, gt),
                "dice": dice_score(pred, gt),
                "hd95": hd95_score(pred, gt),
            })

df = pd.DataFrame(records)
miou = {c: df.loc[df["type"] == c, "iou"].mean() for c in CLASSES}
count = {c: int((df["type"] == c).sum()) for c in CLASSES}
weighted = sum(miou[c] * count[c] for c in CLASSES) / sum(count.values())

metrics = pd.DataFrame({
    "Metric": ["mIoU (Glioma)", "mIoU (Meningioma)", "mIoU (Pituitary)",
               "Weighted mIoU", "Dice", "HD95 (px)"],
    "Value": [miou["Glioma"], miou["Meningioma"], miou["Pituitary"],
              weighted, df["dice"].mean(), np.nanmean(df["hd95"])],
})

print("SABERNET FINAL TEST METRICS")
display(metrics)
metrics.to_csv(os.path.join(DATASET_ROOT, "sabernet_metrics.csv"), index=False)

Test images used: 860
SABERNET FINAL TEST METRICS


,Metric,Value
0,mIoU (Glioma),0.278251
1,mIoU (Meningioma),0.586564
2,mIoU (Pituitary),0.504845
3,Weighted mIoU,0.466997
4,Dice,0.571812
5,HD95 (px),19.092022


In [30]:
for name in ["DEVICE", "MODEL_PATH", "test_pairs",
             "BrainTumorDataset", "LightweightAttentionUNet"]:
    print(name, "->", name in globals())

print("Weights file exists:", os.path.exists(MODEL_PATH))

DEVICE -> True
MODEL_PATH -> True
test_pairs -> True
BrainTumorDataset -> True
LightweightAttentionUNet -> True
Weights file exists: True


In [32]:
import os
import time
import random
import numpy as np
import torch

DATASET_ROOT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "segmentation_task" in dirs:
        DATASET_ROOT = root
        break
if DATASET_ROOT is None:
    raise FileNotFoundError("Add the BRISC 2025 dataset with Add Input.")

seg = os.path.join(DATASET_ROOT, "segmentation_task")
TRAIN_IMAGE_DIR = os.path.join(seg, "train", "images")
TRAIN_MASK_DIR = os.path.join(seg, "train", "masks")
TEST_IMAGE_DIR = os.path.join(seg, "test", "images")
TEST_MASK_DIR = os.path.join(seg, "test", "masks")

IMAGE_SIZE = 256
VAL_SAMPLES = 300
BATCH_SIZE = 16
EPOCHS = 40
LEARNING_RATE = 0.001
BASE = 32
SEED = 42

MODEL_PATH = "/kaggle/working/sabernet_best.pth"

if not torch.cuda.is_available():
    raise RuntimeError("GPU is off. Settings > Accelerator > GPU.")
DEVICE = torch.device("cuda")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Dataset:", DATASET_ROOT)
print("GPU:", torch.cuda.get_device_name(0))

FileNotFoundError: Add the BRISC 2025 dataset with Add Input.